In [ ]:
# Product Review Scraping Pipeline

# This notebook implements a web scraping pipeline that collects product reviews from Alza.cz.

# Pipeline steps:
# 1. Collect product URLs from category pages
# 2. Visit each product page
# 3. Extract product metadata and customer reviews
# 4. Store scraped data in PostgreSQL database

# The scraped dataset will later be processed in the NLP pipeline for sentiment analysis.

In [ ]:
## 1. Environment Setup
## This section initializes required libraries and dependencies for web scraping and database connection.
# Import required libraries

from psycopg2 import connect
from selenium.webdriver import Chrome
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.common.exceptions import InvalidArgumentException
from selenium.webdriver.common.by import By
import time

In [ ]:
## 2. Web Scraping Initialization
## In this section, the Selenium WebDriver is initialized and the target product page is loaded.

# initialize browser
browser = Chrome()

In [ ]:
# List of category pages (pagination) that contain product listings
pages = [f"https://www.alza.cz/apple-watch-11/18918318.htm#f&cud={i}&pg=&prod=&sc=529.5" for i in range(1, 5)]

pages

In [ ]:
# ---------------------------------------
# STEP 1: Collect product page URLs
# ---------------------------------------

products_list = []

for page in pages:

    # Open category page
    browser.get(page)

    time.sleep(2)  # wait for page to load

    # Extract product links from the page
    links = [
        el.get_attribute("href")
        for el in browser.find_elements(
            By.XPATH, '//a[@class="name browsinglink js-box-link"]'
        )
    ]

    # Store links
    products_list.extend(links)

products_list

In [ ]:
# Add review section suffix so the scraper opens the review tab directly
suffix = "#reviews"

products_list = [link + suffix for link in products_list]

products_list

In [ ]:
# ---------------------------------------
# STEP 2: Scrape product metadata
# ---------------------------------------

# Future dataframe columns
product_title = []
productcode = []
review = []


# Iteration, that is one product – values inserted into lists/columns in each iteration, that is one row – each row represents one product
# Iterate through each product page and collect metadata
for product in products_list:
    browser.get(product)
    product_title.append(browser.find_element(By.XPATH, '//h1[@class="h1-placeholder"]').text)
    productcode.append(browser.find_element(By.XPATH,'//span[@data-testid="more-info-product-code"]//span[@data-testid="value"]').text)
    review_text = " ".join(
    t.text.strip()
        for el in browser.find_elements(By.XPATH, '//div[contains(@class,"user-review")]')
        for t in el.find_elements(By.XPATH, './/div[3]//span[contains(@class,"AlzaText")]')
        if t.text.strip()
    )
    review.append(review_text)

print(product_title)
print(productcode)
print(review)


In [ ]:
print(len(product_title))
print(len(productcode))
print(len(review))

Saving scraped data into the PostgreSQL database – table Reviews

In [ ]:
# ---------------------------------------
# STEP 3: Connect to PostgreSQL database
# ---------------------------------------

cnx = connect(
    user="your_USER",
    password="your_PASSWORD",
    host="localhost",
    database="your_DATABASE"
)

cursor = cnx.cursor()


In [ ]:
# ---------------------------------------
# STEP 4: Insert scraped data into database
# ---------------------------------------

cnx.rollback()

for p_name, p_code, p_review in zip(product_title, productcode, review):
    sql = """
        INSERT INTO reviews (product, productcode, review)
        VALUES (%s, %s, %s)
    """
    cursor.execute(sql, (p_name, p_code, p_review))
    
cnx.commit()

NOTES AND ALTERNATIVE CODES

In [ ]:
review_text = ""

elements = browser.find_elements(
    By.XPATH, 
    '//div[contains(@class, "user-review")]'
)

for el in elements:
    reviews = el.find_elements(
        By.XPATH,
        './/div[contains(@class, "reviewsTab-alz-")][3]//span[contains(@class, "AlzaText") and contains(@class,"reviewsTab-alz-")]'
    )

    for r in reviews:
        review_text += r.text + " "

review_text
